# 🚀 NOVA v2.7.0 — Entrenamiento Medio (40k)
**Instrucciones:** Dale a `Entorno de ejecución` -> `Ejecutar todo`.

In [ ]:
# Instalación de librerías
print("⏳ Instalando librerías...")
import os
os.system("pip install unsloth")
os.system("pip install --no-deps trl peft accelerate bitsandbytes datasets")

from unsloth import FastLanguageModel
import torch
from datasets import load_dataset, concatenate_datasets
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. Cargar Modelo
print("⏳ Cargando modelo DeepSeek-Coder 1.3B...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="deepseek-ai/deepseek-coder-1.3b-instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model, r=16, target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16, lora_dropout=0, bias="none", use_gradient_checkpointing="unsloth", random_state=3407,
)

# 2. Preparar Datos (MEDIO: 20k Español + 20k Código = 40k Total)
print("⏳ Descargando datasets (Versión Media - 40k ejemplos)...")
ds1 = load_dataset("somosnlp/somos-clean-alpaca-es", split="train").select(range(20000))
ds2 = load_dataset("iamtarun/code_instructions_120k_alpaca", split="train").select(range(20000))

def format_prompt(e):
    return {"text": f"### Instrucción:\n{e.get('instruction','')}\n\n### Entrada:\n{e.get('input','')}\n\n### Respuesta:\n{e.get('output','')}{tokenizer.eos_token}"}

dataset = concatenate_datasets([ds1, ds2]).shuffle(seed=42).map(format_prompt)

# 3. Entrenar
print(f"🚀 INICIANDO ENTRENAMIENTO MEDIO ({len(dataset)} ejemplos)...")
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dataset, dataset_text_field="text",
    max_seq_length=2048, dataset_num_proc=2, packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=4, warmup_steps=50,
        num_train_epochs=1, learning_rate=2e-4, fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(), logging_steps=50, optim="adamw_8bit",
        weight_decay=0.01, lr_scheduler_type="linear", seed=3407, output_dir="outputs",
    ),
)
stats = trainer.train()

# 4. Guardar y Descargar
print("✅ Entrenamiento terminado. Exportando GGUF...")
model.save_pretrained_gguf("nova-finetuned", tokenizer, quantization_method="q4_k_m")

print("⬇️ DESCARGANDO MODELO AHORA...")
from google.colab import files
try:
    files.download("nova-finetuned/unsloth.Q4_K_M.gguf")
except:
    print("⚠️ Si no se descarga, busca el archivo a la izquierda y descárgalo manual.")